# Component quickstart

This notebook creates a Gemini-backed hypothesis-extraction component and invokes it through the project's registry and configuration patterns. Run the cells from top to bottom.

> **Before you start:** add `GEMINI_API_KEY=...` to `.env` at the repository root. The inference cells make two real Gemini API requests and may incur usage costs.

## 1. Load the project and environment

The path setup works when Jupyter starts in either the repository root or `notebooks/`. `ENVSettings` reads the repository's `.env` file.

In [ ]:
from pathlib import Path
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not find the repository root")


repository_root = find_repository_root(Path.cwd().resolve())
source_dir = repository_root / "src"
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from config.settings import ENVSettings

environment = ENVSettings()
if not environment.gemini_api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY is missing. Add it to .env at the repository root."
    )

print(f"Loaded project and .env from: {repository_root}")

## 2. Create sample input

A `DataSample` contains one question and responses from multiple sources. The extractor will ask Gemini to synthesize a frame of discernment from them.

In [ ]:
from concerted_abduction.types import DataSample, ModelOutput


sample = DataSample(
    scenario_id="quickstart-1",
    created_at="2026-01-01",
    input_message="Why is the plant wilting?",
    outputs=[
        ModelOutput(source="model-a", response="The soil is too dry."),
        ModelOutput(source="model-b", response="The roots may be damaged."),
    ],
)

sample

## 3. Implement and register the component

The component depends on the provider-independent `LanguageModel` interface. Its `from_config()` adapter resolves the configured provider, model, and prompt. The `generate_frame()` method contains the component's actual behavior.

In [ ]:
from typing import Self

from concerted_abduction.fod import (
    FodExtractorRegistry,
    FrameOfDiscernmentExtractor,
)
from concerted_abduction.llm import LanguageModel
from concerted_abduction.types import (
    DataSample,
    FodOutput,
    FrameOfDiscernment,
)
from config.settings import ComponentConfig


@FodExtractorRegistry.register("quickstart_gemini_fod_extractor")
class QuickstartGeminiExtractor(FrameOfDiscernmentExtractor):
    def __init__(self, language_model: LanguageModel) -> None:
        self.language_model = language_model

    @classmethod
    def from_config(cls, cfg: ComponentConfig) -> Self:
        if cfg.llm is None:
            raise ValueError("This extractor requires an llm configuration")
        if cfg.prompt_path is None:
            raise ValueError("This extractor requires a prompt_path")

        from concerted_abduction.llm import LanguageModelRegistry
        from concerted_abduction.utils.utils import load_prompt

        language_model = LanguageModelRegistry.create(
            provider=cfg.llm.provider,
            model=cfg.llm.model_name,
            system_prompt=load_prompt(cfg.prompt_path),
        )
        return cls(language_model=language_model)

    def generate_frame(self, sample: DataSample) -> FodOutput:
        prompt = (
            f"Question: {sample.input_message}\n\nSource responses:\n"
            + "\n".join(
                f"- {output.source}: {output.response}"
                for output in sample.outputs
            )
        )
        frame = self.language_model.generate(
            prompt,
            output_schema=FrameOfDiscernment,
        )
        if not isinstance(frame, FrameOfDiscernment):
            raise TypeError("Language model returned an unexpected output type")

        return FodOutput(
            hypotheses={
                output.source: output.response for output in sample.outputs
            },
            frame=frame,
        )

## 4. Construct and invoke the extractor directly

Direct construction is useful in prototypes and tests. Create the language-model dependency explicitly, pass it to the extractor, and call the component's common method.

In [ ]:
from concerted_abduction.llm import LanguageModelRegistry
from concerted_abduction.utils.utils import load_prompt
from config.settings import ConcertedAbductionSettings


project_config = ConcertedAbductionSettings().fod_extractor
if project_config.llm is None or project_config.prompt_path is None:
    raise ValueError("The hypothesis extractor requires LLM and prompt settings")

language_model = LanguageModelRegistry.create(
    provider=project_config.llm.provider,
    model=project_config.llm.model_name,
    system_prompt=load_prompt(project_config.prompt_path),
)
direct_extractor = QuickstartGeminiExtractor(language_model=language_model)
direct_result = direct_extractor.generate_frame(sample)

assert direct_result.frame.frame
direct_result.model_dump()

## 5. Resolve the component from configuration

The quickstart reuses the project's `fod_extractor` LLM and prompt settings. The registry selects our new class by name, while `from_config()` constructs its Gemini dependency.

In [ ]:
from concerted_abduction.fod import FodExtractorRegistry


extractor_class = FodExtractorRegistry._get("quickstart_gemini_fod_extractor")
extractor = extractor_class.from_config(project_config)

print(f"Component: {type(extractor).__name__}")
print(f"Provider adapter: {type(extractor.language_model).__name__}")
print(f"Model: {extractor.language_model.model}")

## 6. Invoke the registry-resolved component

This cell makes one real Gemini request. The provider asks Gemini for JSON matching `FrameOfDiscernment`, validates it with Pydantic, and returns the component's typed output.

In [ ]:
from concerted_abduction.types import FodOutput


result = extractor.generate_frame(sample)

assert isinstance(result, FodOutput)
assert result.hypotheses == {
    "model-a": "The soil is too dry.",
    "model-b": "The roots may be damaged.",
}
assert result.frame.frame

result.model_dump()

## Where to go next

- Change the prompt assembled in `generate_frame()` and compare the result.
- Pass a `LanguageModel` directly to `QuickstartGeminiExtractor` in a unit test.
- Read `src/concerted_abduction/extractors/llm.py` for the packaged implementation.
- Read `docs/extending.md` before moving a new component into the package.
- Import packaged implementations in their package's `__init__.py` so registration happens before lookup.